Building likelihood maps for a set of representative images.

> TODO: use images from dataset and superimpose the bounding boxes on them

In [1]:
from retinotopy import *

Running on GPU :  NVIDIA GeForce RTX 3060 #GPU= 1
---------------------------------------------------------------------------------------------------
On date 2024-05-24, Running learning on host DESKTOP-27VNO0E with device cuda, pytorch==2.2.0+cu121
---------------------------------------------------------------------------------------------------
Welcome on Linux-5.10.16.3-microsoft-standard-WSL2-x86_64-with-glibc2.35
Random seed 1998 has been set.


In [2]:
# The dataset to import images from
data_set_type = 'full'
args = Params()
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.root_focus  = f'{DATAROOT}/{datetag}_Imagenet_focus' # Directory containing images
args.folders = ['train'] # type of images to use
folder = args.folders[0]

In [3]:
if not(os.path.exists(os.path.join(args.root_focus, folder))):
    os.makedirs(os.path.join(args.root_focus, folder), exist_ok=True)
    for i_img, img_id in enumerate(Imagenet_urls_ILSVRC_2016):
        focus_dir = os.path.join(args.root_focus, folder, img_id)
        os.makedirs(focus_dir, exist_ok=True)

In [21]:
from torchvision.utils import save_image

do_polar = True
model_name = 'resnet101'
model_data_set_type = 'bbox'
args.do_polar = do_polar
print(f'{args.do_polar=}')
print(50*'.')

model_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + '.pt'

model = charge_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()

args.do_saccade = True

image_dataset = image_datasets_transforms(args, shuffle=False, verbose=False)

annotations = get_annotation('csv')
args.seed = np.random.randint(1000)
set_seed(seed=args.seed, seed_torch=True)
df_filename = get_filename(data_cache, datetag, model_data_set_type, model_name, do_polar) + f'_focus_{folder}_{args.seed}.parquet'

print(df_filename)
df_map = None

n_dataset = len(image_dataset)
print(f"Loaded {n_dataset} images")
indices_dataset = np.random.permutation(n_dataset)

with torch.no_grad():
    for ind in tqdm(indices_dataset):
        
        # initialising names
        fname = image_dataset.imgs[ind][0]
        fname_focus = fname.replace(os.path.join(args.root, folder), os.path.join(args.root_focus, folder))
        
        if not(os.path.exists(fname_focus)):
            (image, label) = image_dataset[ind]
            since = time.time()
            image = image.to(device, non_blocking=True)
    
            heat_map = torch.nn.functional.softmax(model(image), dim=1)
    
            image_name = fname.split('/')[-1].split('.')[0]
            heatmap_label = heat_map[:, label]

            arg_max_prior = torch.argmax(heatmap_label)
            position_prior = (arg_max_prior.item()%args.resolution[0], arg_max_prior.item()//args.resolution[1])

            save_image(image[arg_max_prior], fname_focus)  # uncomment to save the image
    
            df_map_ = pd.DataFrame({'ImageId':fname, 'label':label, 'position_prior':[position_prior], 'time':time.time() - since})
            df_map = store_pandas(df_map, df_map_)
    df_map.to_parquet(df_filename)

args.do_polar=True
..................................................
loading .... cached_data/2024-05-24_bbox_resnet101_retino.pt
Random seed 806 has been set.
cached_data/2024-05-24_bbox_resnet101_retino_focus_train_806.parquet
Loaded 1281167 images


  0%|                                                   | 1/1281167 [00:00<276:40:09,  1.29it/s]

/mnt/d/Data/2024-05-24_Imagenet_focus/train/n04311004/n04311004_2735.JPEG


  0%|                                                   | 2/1281167 [00:01<218:57:02,  1.63it/s]

/mnt/d/Data/2024-05-24_Imagenet_focus/train/n02783161/n02783161_20349.JPEG


...

/mnt/d/Data/2024-05-24_Imagenet_focus/train/n02690373/n02690373_9144.JPEG


  0%|                                                  | 30/1281167 [00:15<179:46:04,  1.98it/s]

/mnt/d/Data/2024-05-24_Imagenet_focus/train/n04557648/n04557648_9195.JPEG


  0%|                                                  | 30/1281167 [00:16<193:31:15,  1.84it/s]


KeyboardInterrupt: 